In [ ]:
import pandas as pd  
import os
import glob
from tqdm import tqdm

In [20]:
file_path = r"V:\dunwei\MACE\dataset\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result/"
save_path = file_path + "combine/"

In [21]:
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f'create directory: {save_path}')
else:
    print(f'{save_path} already exists')

V:\dunwei\MACE\dataset\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result/combine/ already exists


In [22]:
dataset = "test"
mode = 2
"""
mode = 1 ---> combine label
mode = 2 ---> combine probability
"""

if mode == 1:
    file_name = f"y_{dataset}_label"
elif mode == 2:
    file_name = f"yhat_{dataset}_probability"
else:
    raise ValueError("mode error")

In [23]:
total_file = os.path.join(file_path, f"y_test_label_*.csv")
file_list = sorted(glob.glob(total_file))
print(len(file_list))

90


In [24]:
data_list = []
for i in tqdm(range(len(file_list)), desc='Loading files'):
    data = pd.read_csv(file_path + f"{file_name}_{i + 1}.csv")
    data_list.append(data)

data_df = pd.concat(data_list, axis=0)
data_df['SortKey'] = data_df.iloc[:, 0].str.extract(r'(\d+)').astype(int)  # 提取數字部分，轉換為數值型
data_df_sorted = data_df.sort_values(by='SortKey').reset_index(drop=True)  # 按數字部分排序
data_df_sorted = data_df_sorted.drop(columns=['SortKey'])  # 刪除輔助列
data_df_sorted.to_csv(save_path + f"{file_name}_combine.csv", index=False)

# 統計每個編號的樣本數
sample_counts = data_df_sorted.iloc[:, 0].value_counts().reset_index()
# 重新命名欄位名稱
sample_counts.columns = ['ID', 'Sample_num']
# 按照 ID 排序（確保 MACE001、MACE002 順序正確）
sample_counts['SortKey'] = sample_counts['ID'].str.extract(r'(\d+)').astype(int)
sample_counts = sample_counts.sort_values(by='SortKey').drop(columns=['SortKey']).reset_index(drop=True)
sample_counts.to_csv(save_path + f"sample_num_{dataset}.csv", index=False)

print(f'\n===========   SAVE INFO   ===========')
print(f"all of files and results saved in {save_path}\n")

Loading files: 100%|██████████| 90/90 [00:00<00:00, 100.52it/s]



===========   SAVE INFO   ===========
all of files and results saved in V:\dunwei\MACE\dataset\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result/combine/

